In [1]:
# Memory safety: cap this kernel to the RAM free right now so an out-of-memory
# feature build fails with a clean MemoryError instead of crashing VS Code /
# thrashing swap. This notebook builds cross-row features (lags/rolling), which
# can't be row-batched, so the guard is the protection here.
import os, sys
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")) ; from memory_compute import *
install_memory_guard()


[memory_guard] hard cap 13.3G virtual on this kernel (total RAM 14.8G, 9.6G free now). Runaway allocations fail cleanly; bounded streaming keeps normal work well under this.


14298148864

In [2]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../")) ; from EPF import variables

REGION        = variables.TARGET_REGION
ALL_REGIONS   = ["nsw", "qld", "vic", "sa"]
OTHER_REGIONS = [r for r in ALL_REGIONS if r != REGION]

REGION_CITY = {"nsw": "sydney", "qld": "brisbane", "vic": "melbourne", "sa": "adelaide"}
CITY = REGION_CITY[REGION]

PER_HOUR = 60 // variables.FEATURE_GRANULARITY_IN_MINUTES   # 12
PER_DAY  = 24 * PER_HOUR                                     # 288
PER_WEEK = 7 * PER_DAY                                       # 2016

# This notebook builds CROSS-SOURCE features: interactions that only make
# sense when two different data sources are combined (demand vs generation-by-
# fuel, gas price vs power price, weather vs load, predispatch forecast vs the
# realised spot). Every input is a value known at (or before) time T -- current
# actuals and causal forecasts -- so all features are leakage-free w.r.t. the
# future-price target.
PROC = "../1_Dataset/Processed_data/"


def _s(frame: pd.DataFrame, name: str) -> pd.Series:
    """Return a column if present, else an all-NaN series (schema-robust)."""
    if name in frame.columns:
        return frame[name].astype("float64")
    return pd.Series(np.nan, index=frame.index, dtype="float64")

In [3]:
# Use the dispatch-price index as the common 5-min spine and left-join every
# other source onto it. Partial sources (predispatch, still regenerating)
# simply leave NaNs outside their available date range. Bounded float32 loads
# keep the joined frame from doubling memory as each source is read.
src = read_parquet_float32(PROC + "1_dispatch_price.parquet", verbose=False)
for _name in ["2_dispatch_region_sum", "3_generation_fuel", "4_STTM_DWGM", "5_weather"]:
    src = src.join(read_parquet_float32(PROC + _name + ".parquet", verbose=False), how="left")

for _opt in ["6_1_predispatch_price"]:
    _path = PROC + _opt + ".parquet"
    if os.path.exists(_path):
        src = src.join(read_parquet_float32(_path, verbose=False), how="left")

df = pd.DataFrame(index=src.index)
df_core_columns = df.columns
print("Aligned source columns:", src.shape[1])
src[:10]


Aligned source columns: 525


,nsw_price,qld_price,sa_price,vic_price,demand_nsw,avail_gen_nsw,interchange_nsw,demand_forecast_nsw,dispatch_gen_nsw,demand_qld,...,predispatch_rrp_vic_h69,predispatch_rrp_vic_h70,predispatch_rrp_vic_h71,predispatch_rrp_vic_h72,predispatch_rrp_vic_h73,predispatch_rrp_vic_h74,predispatch_rrp_vic_h75,predispatch_rrp_vic_h76,predispatch_rrp_vic_h77,predispatch_rrp_vic_h78
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,90.209999,80.270287,103.578568,90.609642,7021.709961,11526.263672,-848.010010,-22.460939,6173.700195,6057.979980,...,53.70079,53.703381,53.7005,53.31879,53.320129,53.324669,53.324669,53.296921,53.296921,53.67939
2018-01-01 00:10:00,95.540657,85.100067,111.817146,96.736847,6952.049805,11531.112305,-819.469971,-29.104000,6132.580078,6131.600098,...,53.70079,53.703381,53.7005,53.31879,53.320129,53.324669,53.324669,53.296921,53.296921,53.67939
2018-01-01 00:15:00,96.293869,85.099998,113.702904,97.344391,6950.970215,11519.082031,-878.929993,-29.913090,6072.040039,6050.430176,...,53.70079,53.703381,53.7005,53.31879,53.320129,53.324669,53.324669,53.296921,53.296921,53.67939
2018-01-01 00:20:00,92.500000,81.728340,106.751778,92.824448,6889.879883,11424.864258,-919.890015,-41.521000,5969.990234,6016.680176,...,53.70079,53.703381,53.7005,53.31879,53.320129,53.324669,53.324669,53.296921,53.296921,53.67939
2018-01-01 00:25:00,92.499901,81.708931,108.531883,92.833130,6846.540039,11424.723633,-898.830017,-36.577148,5947.709961,5993.089844,...,53.70079,53.703381,53.7005,53.31879,53.320129,53.324669,53.324669,53.296921,53.296921,53.67939
2018-01-01 00:30:00,84.092339,73.730026,98.659790,84.389008,6812.330078,11433.789062,-932.390015,-44.602539,5879.939941,5935.270020,...,53.70079,53.703381,53.7005,53.31879,53.320129,53.324669,53.324669,53.296921,53.296921,53.67939
2018-01-01 00:35:00,92.449867,81.088654,105.000053,91.349518,6839.930176,11433.101562,-1020.559998,-29.209961,5819.370117,5965.209961,...,53.70079,53.703381,53.7005,53.31879,53.320129,53.324669,53.324669,53.296921,53.296921,53.67939
2018-01-01 00:40:00,95.790154,84.599960,109.102798,93.914917,6841.149902,11434.149414,-1033.650024,-21.369631,5807.500000,5975.629883,...,53.70079,53.703381,53.7005,53.31879,53.320129,53.324669,53.324669,53.296921,53.296921,53.67939
2018-01-01 00:45:00,90.997910,79.500816,105.000053,89.812172,6794.399902,11436.178711,-1067.010010,-29.372070,5727.390137,5908.029785,...,53.70079,53.703381,53.7005,53.31879,53.320129,53.324669,53.324669,53.296921,53.296921,53.67939


In [4]:
def _add_net_load_features(src: pd.DataFrame) -> pd.DataFrame:
    """
    Load net of variable renewables -- the demand that dispatchable plant must
    actually serve (regionsum demand vs generation-by-fuel). Residual demand and
    its ramp are dominant short-term price drivers (the 'duck curve'), while
    scarcity (residual demand vs available capacity) captures tightness. Mixes
    the dispatch-regionsum and generation-fuel sources.
    """
    out = pd.DataFrame(index=src.index)
    demand      = _s(src, f"demand_{REGION}")
    avail       = _s(src, f"avail_gen_{REGION}")
    interchange = _s(src, f"interchange_{REGION}")
    solar = _s(src, f"solar_mw_{REGION}")
    wind  = _s(src, f"wind_mw_{REGION}")
    coal  = _s(src, f"coal_mw_{REGION}")
    gas_g = _s(src, f"gas_mw_{REGION}")

    vre      = solar + wind
    residual = demand - vre
    out[f"xs_{REGION}_residual_demand"]         = residual.astype(np.float32)
    out[f"xs_{REGION}_residual_demand_ramp_1h"] = residual.diff(PER_HOUR).astype(np.float32)
    out[f"xs_{REGION}_vre_penetration"]         = (vre / (demand + 1)).clip(0, 1).astype(np.float32)
    out[f"xs_{REGION}_net_import_share"]        = (interchange / (demand + 1)).astype(np.float32)
    out[f"xs_{REGION}_scarcity"]                = (residual / (avail + 1)).clip(-1, 2).astype(np.float32)
    out[f"xs_{REGION}_thermal_share"]           = ((coal + gas_g) / (demand + 1)).clip(0, 2).astype(np.float32)
    return out


new_df = _add_net_load_features(src)
df = pd.concat([df, new_df], axis=1)
new_df[:10]

,xs_nsw_residual_demand,xs_nsw_residual_demand_ramp_1h,xs_nsw_vre_penetration,xs_nsw_net_import_share,xs_nsw_scarcity,xs_nsw_thermal_share
Date,,,,,,
2018-01-01 00:05:00,6925.482910,NaN,0.013702,-0.120753,0.600792,0.764176
2018-01-01 00:10:00,6862.376465,NaN,0.012897,-0.117858,0.595067,0.768813
2018-01-01 00:15:00,6867.264648,NaN,0.012041,-0.126429,0.596112,0.759784
2018-01-01 00:20:00,6814.040527,NaN,0.011006,-0.133494,0.596370,0.764637
2018-01-01 00:25:00,6767.945312,NaN,0.011478,-0.131263,0.592343,0.762321
2018-01-01 00:30:00,6728.545410,NaN,0.012297,-0.136848,0.588428,0.760266
2018-01-01 00:35:00,6750.609375,NaN,0.013057,-0.149184,0.590393,0.753654
2018-01-01 00:40:00,6752.479004,NaN,0.012959,-0.151071,0.590502,0.748379
2018-01-01 00:45:00,6710.156738,NaN,0.012397,-0.157019,0.586697,0.756268


In [5]:
def _add_fuel_cost_features(src: pd.DataFrame) -> pd.DataFrame:
    """
    Fuel-cost pressure on price: gas is frequently the marginal fuel in the NEM,
    so the gas price matters most when gas is actually running and when the spot
    price is high relative to fuel cost. Combines the STTM/DWGM gas price, the
    generation-fuel mix and the dispatch spot price. price_to_gas is an implied
    market heat-rate proxy. Current spot at T is known when forecasting T+h.
    """
    out = pd.DataFrame(index=src.index)
    price = _s(src, f"{REGION}_price")
    gasp  = _s(src, f"gas_price_{REGION}")
    gas_g = _s(src, f"gas_mw_{REGION}")
    coal  = _s(src, f"coal_mw_{REGION}")
    hydro = _s(src, f"hydro_mw_{REGION}")
    solar = _s(src, f"solar_mw_{REGION}")
    wind  = _s(src, f"wind_mw_{REGION}")
    biomass = _s(src, f"biomass_mw_{REGION}")
    bdis  = _s(src, f"battery_discharge_mw_{REGION}")

    total_gen = coal + gas_g + hydro + solar + wind + biomass + bdis
    gas_share = gas_g / (total_gen + 1)
    out[f"xs_{REGION}_gas_share_of_gen"]    = gas_share.clip(0, 1).astype(np.float32)
    out[f"xs_{REGION}_gasprice_x_gasshare"] = (gasp * gas_share).astype(np.float32)
    out[f"xs_{REGION}_price_to_gas"]        = (price / (gasp + 1)).clip(-50, 500).astype(np.float32)
    out[f"xs_{REGION}_gas_marginal_signal"] = (gas_share * (price / (gasp + 1)).clip(-50, 500)).astype(np.float32)
    return out


new_df = _add_fuel_cost_features(src)
df = pd.concat([df, new_df], axis=1)
new_df[:10]

,xs_nsw_gas_share_of_gen,xs_nsw_gasprice_x_gasshare,xs_nsw_price_to_gas,xs_nsw_gas_marginal_signal
Date,,,,
2018-01-01 00:05:00,0.000038,0.000210,13.756347,0.000520
2018-01-01 00:10:00,0.000039,0.000217,14.569232,0.000570
2018-01-01 00:15:00,0.000040,0.000220,14.684092,0.000581
2018-01-01 00:20:00,0.000040,0.000221,14.105555,0.000561
2018-01-01 00:25:00,0.000040,0.000223,14.105540,0.000565
2018-01-01 00:30:00,0.000040,0.000224,12.823450,0.000517
2018-01-01 00:35:00,0.000040,0.000225,14.097910,0.000571
2018-01-01 00:40:00,0.000041,0.000230,14.607279,0.000603
2018-01-01 00:45:00,0.000042,0.000231,13.876497,0.000577


In [6]:
def _add_weather_load_features(src: pd.DataFrame) -> pd.DataFrame:
    """
    Temperature-driven load pressure: cooling / heating degree-days for the
    region's reference city interacted with demand. Hot afternoons combined with
    already-high demand are the classic NEM price-spike setup. Combines the
    weather and dispatch-regionsum sources. Weather is legitimately known ahead
    of time, and demand at T is known when forecasting T+h.
    """
    out = pd.DataFrame(index=src.index)
    temp   = _s(src, f"temp_{CITY}")
    demand = _s(src, f"demand_{REGION}")

    cdd = (temp - 18.0).clip(lower=0)
    hdd = (18.0 - temp).clip(lower=0)
    demand_rank = demand.rolling(PER_WEEK, min_periods=PER_HOUR).rank(pct=True)

    out[f"xs_{REGION}_cdd_x_demand"]     = (cdd * demand).astype(np.float32)
    out[f"xs_{REGION}_hdd_x_demand"]     = (hdd * demand).astype(np.float32)
    out[f"xs_{REGION}_hot_high_demand"]  = (cdd * demand_rank).astype(np.float32)
    out[f"xs_{REGION}_cold_high_demand"] = (hdd * demand_rank).astype(np.float32)
    return out


new_df = _add_weather_load_features(src)
df = pd.concat([df, new_df], axis=1)
new_df[:10]

,xs_nsw_cdd_x_demand,xs_nsw_hdd_x_demand,xs_nsw_hot_high_demand,xs_nsw_cold_high_demand
Date,,,,
2018-01-01 00:05:00,30778.494141,0.0,NaN,NaN
2018-01-01 00:10:00,30357.285156,0.0,NaN,NaN
2018-01-01 00:15:00,30236.722656,0.0,NaN,NaN
2018-01-01 00:20:00,29856.150391,0.0,NaN,NaN
2018-01-01 00:25:00,29554.236328,0.0,NaN,NaN
2018-01-01 00:30:00,29293.013672,0.0,NaN,NaN
2018-01-01 00:35:00,29297.697266,0.0,NaN,NaN
2018-01-01 00:40:00,29188.904297,0.0,NaN,NaN
2018-01-01 00:45:00,28876.199219,0.0,NaN,NaN


In [7]:
def _add_forward_signal_features(src: pd.DataFrame) -> pd.DataFrame:
    """
    Forward-vs-spot signals: the predispatch RRP forecast for the next interval
    (predispatch_rrp_{R}_h1, a causal forecast known at T) minus / over the
    realised spot at T. A positive squeeze means the market expects prices to
    firm. Combines the predispatch-price and dispatch-price sources. All inputs
    are known at T, so this is leakage-free. Guarded in case predispatch data is
    not yet regenerated.
    """
    out = pd.DataFrame(index=src.index)
    price = _s(src, f"{REGION}_price")
    pd_h1 = _s(src, f"predispatch_rrp_{REGION}_h1")

    out[f"xs_{REGION}_fwd_price_squeeze"] = (pd_h1 - price).astype(np.float32)
    out[f"xs_{REGION}_fwd_price_ratio"]   = (pd_h1 / (price.abs() + 1)).clip(-50, 500).astype(np.float32)
    return out


new_df = _add_forward_signal_features(src)
df = pd.concat([df, new_df], axis=1)
new_df[:10]

,xs_nsw_fwd_price_squeeze,xs_nsw_fwd_price_ratio
Date,,
2018-01-01 00:05:00,5.580002,1.050214
2018-01-01 00:10:00,0.249344,0.992224
2018-01-01 00:15:00,-0.503868,0.984543
2018-01-01 00:20:00,3.290001,1.024492
2018-01-01 00:25:00,3.290100,1.024493
2018-01-01 00:30:00,8.407661,1.087054
2018-01-01 00:35:00,0.050133,0.989836
2018-01-01 00:40:00,-3.290154,0.955676
2018-01-01 00:45:00,1.502090,1.005458


In [8]:
print("Total features:", df.shape[1])
df = df.drop(columns=df_core_columns)
df.to_parquet("../2_Features_build/Feature_data/10_cross_source.parquet")
df.shape

Total features: 16


(893664, 16)

In [9]:
# Free this kernel's memory so the next notebook has RAM to work with
# (clears data variables + returns freed heap to the OS).
release_memory()


[release_memory] cleared 18 variable(s); kernel rss 2.20G, 7.6G RAM free now
